In [ ]:
"""
简单的hash_html去重脚本

使用方法：
1. 直接调用函数：
   hash_dedup_simple(spark, input_base_path, output_base_path, config)

2. 参数说明：
   - input_base_path: 输入数据路径
   - output_base_path: 输出路径前缀
   - config: xinghe配置对象

3. 输出：
   - 对hash_html字段进行全量去重
   - 提取file_path后缀（如20241109/xxx.jsonl.gz），拼接output_base_path
   - 使用S3DocWriter在partition内独立写文件，避免collect操作

4. 数据格式：
   输入: value包含{"sub_path", "hash_html", "track_id", "file_path"}
   输出: 按file_path后缀分组保存到output_base_path/后缀路径
"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from xinghe.spark import *
from xinghe.s3 import *
import json
import uuid
import traceback
from datetime import datetime

# 错误日志路径
ERROR_PATH = "s3://qa-huawei/chupei/cc-domain-centric-store/error_logs/"

# 异常日志
def get_s3_doctor(target_theme):
    partition_id = str(uuid.uuid4())
    current_time = datetime.now().strftime("%Y%m%d")
    error_log_path = f"{ERROR_PATH}{target_theme}/{current_time}/{partition_id}.jsonl"
    s3_doc_writer = S3DocWriter(path=error_log_path)
    return s3_doc_writer


def write_partition_to_s3(iterator, output_base_path):
    """
    在partition内使用S3DocWriter写文件
    """    
    # 初始化错误日志写入器
    s3_doc_writer = get_s3_doctor("dedup_sec")
    total_records = 0
    
    try:
        # 按output_path分组数据
        file_groups = {}
        for row in iterator:
            output_path = row.output_path
            if output_path not in file_groups:
                file_groups[output_path] = []
            file_groups[output_path].append(row.output_value)
            total_records += 1
        
        # 为每个文件写入数据
        for output_path, values in file_groups.items():
            try:
                writer = S3DocWriter(output_path)
                for value in values:
                    # 将JSON字符串转换回字典，S3DocWriter会自动添加换行符
                    data_dict = json.loads(value)
                    writer.write(data_dict)
                writer.flush()
                print(f"成功写入文件: {output_path}, 记录数: {len(values)}")
            except Exception as e:
                print(f"写入文件失败: {output_path}, 错误: {str(e)}")
                
    except Exception as e:
        # 外层异常捕获，记录到错误日志
        error_info = {
            "error_type": type(e).__name__,
            "error_message": str(e),
            "traceback": traceback.format_exc(),
            "input_data": "partition_processing_error",
            "stage": "write_partition_to_s3",
            "timestamp": datetime.now().isoformat()
        }
        s3_doc_writer.write(error_info)
        s3_doc_writer.flush()
        print(f"分区处理失败: {str(e)}")
    
    return iter([total_records])  # 返回该分区处理的记录数

def hash_dedup_simple(spark, input_base_path, output_base_path, config):
    """
    简单的hash_html去重处理
    
    Args:
        spark: SparkSession
        input_base_path: 输入数据路径
        output_base_path: 输出路径前缀
        config: xinghe配置对象
    """
    print(f"开始处理hash_html去重...")
    print(f"输入路径: {input_base_path}")
    print(f"输出路径前缀: {output_base_path}")
    
    # 使用read_any_path读取数据
    input_df = read_any_path(spark, input_base_path, config)
    
    # 解析JSON数据
    parsed_df = input_df.select(
        from_json(col("value"), StructType([
            StructField("sub_path", StringType()),
            StructField("hash_html", StringType()),
            StructField("track_id", StringType()),
            StructField("file_path", StringType())
        ])).alias("data")
    ).select(
        col("data.*")
    ).filter(col("hash_html").isNotNull())
    
    print("开始全量去重...")
    # 全量去重
    deduped_df = parsed_df.dropDuplicates(["hash_html"])
    
    # 构造输出路径和数据
    result_df = deduped_df.withColumn(
        "output_path",
        concat(
            lit(output_base_path + "/"),
            regexp_extract(col("file_path"), r".*/([^/]+/[^/]+\.jsonl(?:\.gz)?)$", 1)
        )
    ).withColumn(
        "output_value",
        to_json(struct("sub_path", "hash_html", "track_id", "file_path"))
    )
    
    print("开始写入文件...")
    # 使用mapPartitions在每个partition内独立写文件，避免collect
    # 按output_path分区，确保同一output_path的数据在同一partition
    total_written_records = result_df.repartition(col("output_path")) \
                                    .rdd \
                                    .mapPartitions(lambda iterator: write_partition_to_s3(iterator, output_base_path)) \
                                    .sum()  # 对所有分区的记录数求和
    
    print(f"处理完成! 写入的总记录数: {total_written_records}")
    return deduped_df

def hash_dedup_incremental(spark, input_base_path, output_base_path, existing_base_path, config):
    """
    增量hash_html去重处理
    
    Args:
        spark: SparkSession
        input_base_path: 新增数据路径 (如 v6)
        output_base_path: 输出路径前缀 (如 v2)
        existing_base_path: 已存在的去重数据路径 (如 v2，用于读取已有hash_html)
        config: xinghe配置对象
    """
    print(f"开始处理增量hash_html去重...")
    print(f"新增数据路径: {input_base_path}")
    print(f"已存在数据路径: {existing_base_path}")
    print(f"输出路径前缀: {output_base_path}")
    
    # 1. 读取新增数据
    print("读取新增数据...")
    new_input_df = read_any_path(spark, input_base_path, config)
    
    # 解析新增数据
    new_parsed_df = new_input_df.select(
        from_json(col("value"), StructType([
            StructField("sub_path", StringType()),
            StructField("hash_html", StringType()),
            StructField("track_id", StringType()),
            StructField("file_path", StringType())
        ])).alias("data")
    ).select(
        col("data.*")
    ).filter(col("hash_html").isNotNull())
    
    print("新增数据读取完成")
    
    # 2. 高效去重策略：避免读取全量已存在数据
    print("开始高效增量去重...")
    
    # 策略：直接对新数据进行去重，然后使用left_anti join过滤
    # 先对新数据内部去重，减少需要join的数据量
    print("对新数据进行内部去重...")
    new_internal_deduped = new_parsed_df.dropDuplicates(["hash_html"])
    print("新数据内部去重完成")
    
    # 只读取已存在数据的hash_html字段，避免读取完整记录
    print("读取已存在数据的hash_html...")
    try:
        existing_df = read_any_path(spark, existing_base_path, config)
        # 直接提取hash_html，避免解析完整JSON
        existing_hashes = existing_df.select(
            get_json_object(col("value"), "$.hash_html").alias("hash_html")
        ).filter(col("hash_html").isNotNull())
        
        print("已存在hash_html读取完成")
        
        # 使用分区优化的join，确保相同hash在同一分区
        print("执行高效去重join...")
        new_unique_df = new_internal_deduped.repartition(col("hash_html")) \
                                           .join(existing_hashes.repartition(col("hash_html")), 
                                                ["hash_html"], "left_anti")
        
    except Exception as e:
        print(f"读取已存在数据失败（可能是首次运行）: {str(e)}")
        print("请使用 hash_dedup_simple 函数进行首次全量去重")
        raise e
    
    # 4. 过滤完成
    print("高效去重完成...")
    
    print("重复数据过滤完成")
    
    # 5. 构造输出路径和数据（new_unique_df已经是去重后的数据）
    result_df = new_unique_df.withColumn(
        "output_path",
        concat(
            lit(output_base_path + "/"),
            regexp_extract(col("file_path"), r".*/([^/]+/[^/]+\.jsonl(?:\.gz)?)$", 1)
        )
    ).withColumn(
        "output_value",
        to_json(struct("sub_path", "hash_html", "track_id", "file_path"))
    )
    
    print("开始写入新增数据...")
    # 使用mapPartitions在每个partition内独立写文件，和hash_dedup_simple一致
    total_written_records = result_df.repartition(col("output_path")) \
                                    .rdd \
                                    .mapPartitions(lambda iterator: write_partition_to_s3(iterator, output_base_path)) \
                                    .sum()  # 对所有分区的记录数求和
    
    print(f"增量去重处理完成! 写入的总记录数: {total_written_records}")
    return new_unique_df



# 使用示例
if __name__ == "__main__":
    
    # 配置
    config = {
        "spark_conf_name": "spark_4",
        "skip_success_check": True,
        "spark.yarn.queue": "pipeline.clean",
        "spark.dynamicAllocation.maxExecutors": 1000, # 控制1万并发
        "spark.executor.memory": "80g",
        "spark.executor.memoryOverhead": "40g",  # 增加到40GB
        # "spark.speculation": "true",     # 启用推测执行
        # "maxRecordsPerFile": 200000,      # 增加每文件记录数以减少总文件数
        "output_compression": "gz",
        "skip_output_version": True,
        "skip_output_check": True,
        "spark.sql.shuffle.partitions": "10000",  # 减少分区数
        "spark.default.parallelism": "10000",
        
        # Shuffle 优化配置
        "spark.shuffle.io.maxRetries": "10",  # 增加shuffle重试次数
        "spark.shuffle.io.retryWait": "30s",  # 重试等待时间
        "spark.shuffle.compress": "true",  # 启用shuffle压缩
        "spark.shuffle.spill.compress": "true",  # 启用spill压缩
        
        # 网络和超时配置
        "spark.network.timeout": "3600s",  # 进一步增加网络超时
        "spark.broadcast.timeout": "3600s", 
        "spark.broadcast.compress": "true",
        "spark.rpc.askTimeout": "3600s",   
        "spark.rpc.lookupTimeout": "3600s", 
        "spark.storage.blockManagerSlaveTimeoutMs": "3600000",
        
    }
    spark = new_spark_session("cc_dumps.dedup.sec", config)
    sc = spark.sparkContext
    sc.setLogLevel("ERROR")
    sc
    
    # 示例用法
    
    # 方案1: 全量去重（首次运行）
    # input_base_path = "s3://web-parse-hw60p/PJCC-dedup/hash/v5"
    # output_base_path = "s3://web-parse-hw60p/PJCC-dedup/hash-dedup/v2"
    # result = hash_dedup_simple(spark, input_base_path, output_base_path, config)
    
    # 方案2: 增量去重（后续运行）
    input_base_path = "s3://web-parse-hw60p/PJCC-dedup/hash/v6"  # 新增数据
    output_base_path = "s3://web-parse-hw60p/PJCC-dedup/hash-dedup/v2"  # 输出路径
    existing_base_path = "s3://web-parse-hw60p/PJCC-dedup/hash-dedup/v2"  # 已存在数据路径
    
    # 执行增量去重
    result = hash_dedup_incremental(spark, input_base_path, output_base_path, existing_base_path, config)
    
    spark.stop() 

In [ ]:
# Spark配置
config = {
    "spark_conf_name": "spark_4",
    "skip_success_check": True,
    "spark.yarn.queue": "pipeline.clean",
    "spark.dynamicAllocation.maxExecutors": 2000,  # 控制1万并发
    "output_compression": "gz",
    "skip_output_version": True,
    "skip_output_check": True,
    "spark.sql.shuffle.partitions": "20000",
    "spark.default.parallelism": "20000",
    "spark.network.timeout": "1200s",  # 网络超时
    "spark.broadcast.timeout": "1800s",  # 增加广播超时
    "spark.broadcast.compress": "true",  # 确保广播压缩
    "spark.task.maxFailures": 8,
}
    
from xinghe.spark.session_ext import new_spark_session   
from xinghe.spark.read_ext import read_any_path
spark = new_spark_session("extract_unique_file_paths", config)
sc = spark.sparkContext
sc.setLogLevel("ERROR")
input_path = "s3://web-parse-hw60p/PJCC-dedup/hash-dedup/v2/"
input_df = read_any_path(spark, input_path, config)
    
total_records = input_df.count()
print(f"总记录数: {total_records:,}")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from xinghe.spark import *
from xinghe.s3 import *
import json
import uuid
import traceback
from datetime import datetime

input_base_path = "s3://web-parse-hw60p/PJCC-dedup/hash/v5/20230801"
input_df = read_any_path(spark, input_base_path, config)
input_df.count()

In [ ]:
input_base_path = "s3://web-parse-hw60p/PJCC-dedup/hash_dedup/v1/20230801"
input_df = read_any_path(spark, input_base_path, config)
input_df.count()

In [ ]:
files = list(list_s3_objects('s3://web-parse-hw60p/PJCC-dedup/hash/v5/20230801/', is_prefix=True))
print(len(files))
files = list(list_s3_objects('s3://web-parse-hw60p/PJCC-dedup/hash_dedup/v1/20230801/', is_prefix=True))
print(len(files))